In [2]:
import pandas as pd

train = pd.read_parquet("../data_processed/train.parquet")
val = pd.read_parquet("../data_processed/val.parquet")
test = pd.read_parquet("../data_processed/test.parquet")
future_2026 = pd.read_parquet("../data_processed/future_2026.parquet")

print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)
print("2026:", future_2026.shape)

Train: (24200, 42)
Val: (1200, 42)
Test: (1838, 42)
2026: (198, 42)


In [3]:
print(train.columns.tolist())

['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'name', 'date', 'driverRef', 'forename', 'surname', 'nationality', 'constructorRef', 'name_constructor', 'nationality_constructor', 'status', 'quali_position', 'grid_fixed', 'feat_grid', 'feat_driver_form_last3', 'feat_team_form_last3', 'feat_circuit_history', 'dnf', 'feat_driver_dnf_rate_last5', 'prev_points', 'prev_standing_position', 'podium']


In [4]:
for split_df in [train, val, test, future_2026]:
    split_df['podium'] = (split_df['positionOrder'] <= 3).astype(int)

print(train['podium'].mean(), val['podium'].mean(), test['podium'].mean())

0.12450413223140495 0.15 0.1501632208922742


In [5]:
train.to_parquet("../data_processed/train.parquet", index=False)
val.to_parquet("../data_processed/val.parquet", index=False)
test.to_parquet("../data_processed/test.parquet", index=False)
future_2026.to_parquet("../data_processed/future_2026.parquet", index=False)
print("Re-saved with podium column included.")

Re-saved with podium column included.


In [6]:
feature_cols = [
    'feat_grid', 'feat_driver_form_last3', 'feat_team_form_last3',
    'feat_circuit_history', 'feat_driver_dnf_rate_last5',
    'prev_points', 'prev_standing_position'
]

X_train, y_train = train[feature_cols], train['podium']
X_val, y_val = val[feature_cols], val['podium']
X_test, y_test = test[feature_cols], test['podium']

print(y_train.mean(), y_val.mean(), y_test.mean())

0.12450413223140495 0.15 0.1501632208922742


In [7]:
import sys
!{sys.executable} -m pip install xgboost


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # handles class imbalance
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained.")

Model trained.


In [9]:
from sklearn.metrics import roc_auc_score, log_loss, classification_report

val_probs = model.predict_proba(X_val)[:, 1]
val_preds = model.predict(X_val)

print("AUC-ROC:", roc_auc_score(y_val, val_probs))
print("Log Loss:", log_loss(y_val, val_probs))
print("\nClassification Report:\n", classification_report(y_val, val_preds))

AUC-ROC: 0.9307244008714597
Log Loss: 0.3870920240879059

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.81      0.89      1020
           1       0.46      0.92      0.61       180

    accuracy                           0.83      1200
   macro avg       0.72      0.86      0.75      1200
weighted avg       0.90      0.83      0.85      1200



In [10]:
val_eval = val.copy()
val_eval['pred_prob'] = val_probs

def top3_accuracy(df_eval):
    correct = 0
    total = 0
    for race_id, group in df_eval.groupby('raceId'):
        actual_podium = set(group[group['podium'] == 1]['driverId'])
        predicted_podium = set(group.nlargest(3, 'pred_prob')['driverId'])
        correct += len(actual_podium & predicted_podium)
        total += len(actual_podium)
    return correct / total

print("Top-3 match rate:", top3_accuracy(val_eval))

Top-3 match rate: 0.7


In [11]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance)

feat_grid                     0.751418
prev_standing_position        0.112087
feat_team_form_last3          0.034976
prev_points                   0.032071
feat_driver_dnf_rate_last5    0.028552
feat_driver_form_last3        0.020866
feat_circuit_history          0.020030
dtype: float32


In [12]:
val_eval = val.copy()
val_eval['pred_prob'] = val_probs

def top3_accuracy(df_eval):
    correct = 0
    total = 0
    for race_id, group in df_eval.groupby('raceId'):
        actual_podium = set(group[group['podium'] == 1]['driverId'])
        predicted_podium = set(group.nlargest(3, 'pred_prob')['driverId'])
        correct += len(actual_podium & predicted_podium)
        total += len(actual_podium)
    return correct / total

print("Top-3 match rate:", top3_accuracy(val_eval))

Top-3 match rate: 0.7


In [14]:
#what if we just always predicted the top-3 grid positions as the podium
val_eval['grid_rank'] = val_eval.groupby('raceId')['feat_grid'].rank(method='first')

def top3_accuracy_baseline(df_eval, rank_col):
    correct = 0
    total = 0
    for race_id, group in df_eval.groupby('raceId'):
        actual_podium = set(group[group['podium'] == 1]['driverId'])
        predicted_podium = set(group.nsmallest(3, rank_col)['driverId'])
        correct += len(actual_podium & predicted_podium)
        total += len(actual_podium)
    return correct / total

print("Baseline (top-3 grid positions) match rate:", top3_accuracy_baseline(val_eval, 'grid_rank'))

Baseline (top-3 grid positions) match rate: 0.7166666666666667


Identify "surprise" races

In [15]:
def classify_races(df_eval):
    surprise_race_ids = []
    normal_race_ids = []
    for race_id, group in df_eval.groupby('raceId'):
        actual_podium = set(group[group['podium'] == 1]['driverId'])
        grid_top3 = set(group.nsmallest(3, 'grid_rank')['driverId'])
        if actual_podium == grid_top3:
            normal_race_ids.append(race_id)
        else:
            surprise_race_ids.append(race_id)
    return surprise_race_ids, normal_race_ids

surprise_ids, normal_ids = classify_races(val_eval)
print(f"Surprise races: {len(surprise_ids)}, Normal races: {len(normal_ids)}, Total: {len(surprise_ids) + len(normal_ids)}")

Surprise races: 40, Normal races: 20, Total: 60


In [16]:
def top3_accuracy_on_subset(df_eval, race_ids, rank_or_prob_col, use_smallest=False):
    subset = df_eval[df_eval['raceId'].isin(race_ids)]
    correct = 0
    total = 0
    for race_id, group in subset.groupby('raceId'):
        actual_podium = set(group[group['podium'] == 1]['driverId'])
        if use_smallest:
            predicted_podium = set(group.nsmallest(3, rank_or_prob_col)['driverId'])
        else:
            predicted_podium = set(group.nlargest(3, rank_or_prob_col)['driverId'])
        correct += len(actual_podium & predicted_podium)
        total += len(actual_podium)
    return correct / total if total > 0 else None

model_surprise = top3_accuracy_on_subset(val_eval, surprise_ids, 'pred_prob', use_smallest=False)
baseline_surprise = top3_accuracy_on_subset(val_eval, surprise_ids, 'grid_rank', use_smallest=True)

print("Model match rate on SURPRISE races:", model_surprise)
print("Baseline match rate on SURPRISE races:", baseline_surprise)

Model match rate on SURPRISE races: 0.5583333333333333
Baseline match rate on SURPRISE races: 0.575


train a model without feat_grid

In [17]:
feature_cols_no_grid = [
    'feat_driver_form_last3', 'feat_team_form_last3',
    'feat_circuit_history', 'feat_driver_dnf_rate_last5',
    'prev_points', 'prev_standing_position'
]

X_train_ng = train[feature_cols_no_grid]
X_val_ng = val[feature_cols_no_grid]

model_no_grid = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='logloss',
    random_state=42
)

model_no_grid.fit(X_train_ng, y_train)

val_probs_no_grid = model_no_grid.predict_proba(X_val_ng)[:, 1]

print("AUC-ROC (no grid):", roc_auc_score(y_val, val_probs_no_grid))

AUC-ROC (no grid): 0.9120697167755992


In [18]:
val_eval['pred_prob_no_grid'] = val_probs_no_grid

overall_no_grid = top3_accuracy_on_subset(val_eval, val_eval['raceId'].unique(), 'pred_prob_no_grid', use_smallest=False)
surprise_no_grid = top3_accuracy_on_subset(val_eval, surprise_ids, 'pred_prob_no_grid', use_smallest=False)

print("No-grid model — overall top-3 match rate:", overall_no_grid)
print("No-grid model — surprise-races top-3 match rate:", surprise_no_grid)

No-grid model — overall top-3 match rate: 0.6333333333333333
No-grid model — surprise-races top-3 match rate: 0.5583333333333333


tuning pass

In [19]:
from itertools import product

param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.05, 0.1],
}

results_list = []

for n_est, depth, lr in product(param_grid['n_estimators'], param_grid['max_depth'], param_grid['learning_rate']):
    m = XGBClassifier(
        n_estimators=n_est,
        max_depth=depth,
        learning_rate=lr,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric='logloss',
        random_state=42
    )
    m.fit(X_train, y_train)
    probs = m.predict_proba(X_val)[:, 1]
    
    val_eval_temp = val.copy()
    val_eval_temp['pred_prob'] = probs
    top3_rate = top3_accuracy(val_eval_temp)
    auc = roc_auc_score(y_val, probs)
    
    results_list.append({
        'n_estimators': n_est, 'max_depth': depth, 'learning_rate': lr,
        'auc': auc, 'top3_rate': top3_rate
    })

results_df = pd.DataFrame(results_list).sort_values('top3_rate', ascending=False)
print(results_df.head(10))

    n_estimators  max_depth  learning_rate       auc  top3_rate
1            100          3           0.05  0.935147   0.727778
0            100          3           0.01  0.934333   0.722222
3            100          4           0.01  0.933595   0.722222
9            200          3           0.01  0.934975   0.722222
4            100          4           0.05  0.933156   0.722222
18           400          3           0.01  0.935098   0.722222
12           200          4           0.01  0.934387   0.722222
21           400          4           0.01  0.934474   0.716667
15           200          6           0.01  0.932282   0.716667
6            100          6           0.01  0.929581   0.716667


In [20]:
best_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='logloss',
    random_state=42
)

best_model.fit(X_train, y_train)
val_probs_best = best_model.predict_proba(X_val)[:, 1]

val_eval['pred_prob_best'] = val_probs_best

overall_best = top3_accuracy_on_subset(val_eval, val_eval['raceId'].unique(), 'pred_prob_best', use_smallest=False)
surprise_best = top3_accuracy_on_subset(val_eval, surprise_ids, 'pred_prob_best', use_smallest=False)

print("Tuned model — overall top-3 match rate:", overall_best)
print("Tuned model — surprise-races top-3 match rate:", surprise_best)
print("Baseline (grid only) — surprise-races match rate:", baseline_surprise)

Tuned model — overall top-3 match rate: 0.7277777777777777
Tuned model — surprise-races top-3 match rate: 0.5916666666666667
Baseline (grid only) — surprise-races match rate: 0.575


In [21]:
test_probs = best_model.predict_proba(X_test)[:, 1]

print("TEST SET — AUC-ROC:", roc_auc_score(y_test, test_probs))
print("TEST SET — Log Loss:", log_loss(y_test, test_probs))

test_eval = test.copy()
test_eval['pred_prob'] = test_probs
test_eval['grid_rank'] = test_eval.groupby('raceId')['feat_grid'].rank(method='first')

test_top3 = top3_accuracy(test_eval)
test_baseline = top3_accuracy_on_subset(test_eval, test_eval['raceId'].unique(), 'grid_rank', use_smallest=True)

print("TEST SET — Model top-3 match rate:", test_top3)
print("TEST SET — Baseline (grid) top-3 match rate:", test_baseline)

TEST SET — AUC-ROC: 0.9400086288481879
TEST SET — Log Loss: 0.4295596480369568
TEST SET — Model top-3 match rate: 0.6739130434782609
TEST SET — Baseline (grid) top-3 match rate: 0.6702898550724637


In [22]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/podium_model.pkl")
print("Model saved.")

Model saved.
